# TransAxx ViT sweep — ImageNet × {ViT models} × multipliers (eval-only)

Approximates a Vision Transformer by swapping every `nn.Linear` (QKV / projection / MLP) for
`AdaPT_Linear` carrying one of our `fpga_ish` multipliers, then evaluating on ImageNet val.

**Scope notes**
- **Eval-only** (`RETRAIN_EPOCHS = 0`). ImageNet retraining needs the full train set, so it's out of scope.
- **Coverage:** approximates the **Linear** layers (the bulk of ViT FLOPs). The functional
  attention-score matmuls (`q@kᵀ`, `attn@v`) stay exact.
- **Transform:** loaders use **each model's own timm transform** (correct mean/std/crop/interp), since a
  generic ImageNet transform under-feeds models like `vit_small` that expect (0.5, 0.5, 0.5).
- **Baselines (read results this way):** each model gets a plain **`fp32` row** (no quantization) and a
  **`mul8s_acc` row** (8-bit, exact multiplier). `fp32` vs the model's published number = the val-set's
  cost; `mul8s_acc` vs `fp32` = the int8 PTQ cost (TransAxx's per-tensor scheme costs 2–3 pp even in its
  own paper); each `ish` row vs `mul8s_acc` = the approximate multiplier's cost (the quantity of interest).

**Val set:** the real **ILSVRC2012 validation set**. Stage it once with the **Prepare real ImageNet**
cell below (needs the two `.tar` files from image-net.org). Re-running the sweep is safe — rows already
in the CSV are skipped.

In [1]:
import csv, gc, os, time, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import timm

from classification.utils import (replace_linear_layers, collect_stats, compute_amax, AdaPT_Linear)
from classification.train import train_one_epoch

device = "cuda"
assert torch.cuda.is_available(), "CUDA not available — check nvidia-container-toolkit / CDI override"


## Sweep config

**Models / parameters are set in `0_configure_sweeps.ipynb`** (edit there, Run All, then re-run this
notebook, since the cell below just reads its `sweep_config.json`). `MODELS` can be any
ImageNet-pretrained timm name. Results land in `results/runs.csv` (host:
`docker/results/transaxx/runs.csv`). `FRESH=True` lives here (it deletes results) and drops **only this
notebook's** rows (`MODELS` on ImageNet).

In [2]:
import json as _json

IMAGENET_DIR = "/workspace/datasets/imagenet_data"   # holds the imagenet/ val + calib

# What to sweep comes from 0_configure_sweeps' manifest (EDIT THERE, not here).
# The literals below are only fallbacks for when the manifest hasn't been written yet.
_cfg = (_json.loads(Path("sweep_config.json").read_text()).get("transaxx_vit", {})
        if Path("sweep_config.json").exists() else {})
DATASET_NAME   = _cfg.get("dataset", "imagenet")   # real ILSVRC2012 val
MODELS         = _cfg.get("models", ["vit_small_patch16_224", "deit_small_patch16_224",
                                     "vit_base_patch16_224"])
RETRAIN_EPOCHS = _cfg.get("retrain_epochs", 0)   # eval-only; >0 needs full ImageNet train
BATCH_SIZE     = _cfg.get("batch_size", 128)
CALIB_BATCHES  = _cfg.get("calib_batches", 8)    # ViTs need more calibration than CNNs

# Multipliers come from 0_build_multipliers' manifest (define once, sweep everywhere).
# mul8s_exact is the generator's exact twin of the mul8s_acc baseline, so we keep acc
# and drop the duplicate. Falls back to the fixed list if the manifest is not built yet.
_manifest = Path("../luts/multipliers.json")
if _manifest.exists():
    _built = _json.loads(_manifest.read_text())["multipliers"]
    MULTS = [m for m in _built if m != "mul8s_exact"] + ["mul8s_acc"]
else:
    MULTS = [f"mul8s_fpga_ish_{i}" for i in range(1, 6)] + ["mul8s_acc"]

NUM_WORKERS = 8
N_CLASSES = 1000

# FRESH=False -> resume (skip rows already in runs.csv).
# FRESH=True  -> drop ONLY this notebook's rows (dataset=imagenet & model in MODELS) +
#                their confusion matrices, then recompute. Never touches CIFAR rows.
FRESH = False

# Jupyter roots at /work, and this notebook lives in /work/notebooks, so results go to
# /work/results/transaxx (compose mounts ./results -> /work/results).
RESULTS_DIR = Path("../results/transaxx")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_DIR / "runs.csv"
COLS = ["run_id", "timestamp", "dataset", "model", "multiplier",
        "retrained_epochs", "top1", "top5", "n_samples", "runtime_s", "notes"]

def cm_path(dataset, model_name, mult, retrained_epochs):
    short = mult.replace("mul8s_", "")
    d = RESULTS_DIR / "confusion_matrices" / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{model_name}_{short}_e{retrained_epochs}.npy"

if FRESH and CSV_PATH.exists():
    _df = pd.read_csv(CSV_PATH)
    _mask = (_df.dataset == DATASET_NAME) & (_df.model.isin(MODELS))
    for _, _r in _df[_mask].iterrows():
        cm_path(_r.dataset, _r.model, _r.multiplier, _r.retrained_epochs).unlink(missing_ok=True)
    _df[~_mask].to_csv(CSV_PATH, index=False)
    print(f"FRESH=True -> dropped {int(_mask.sum())} existing rows for {MODELS} on {DATASET_NAME}")

_variants = 1 if RETRAIN_EPOCHS == 0 else 2
print(f"Matrix: 1 dataset x {len(MODELS)} models x ({len(MULTS)} mults x {_variants} variant(s) "
      f"+ 1 fp32 baseline) = {len(MODELS) * (len(MULTS) * _variants + 1)} rows")

Matrix: 1 dataset x 3 models x (6 mults x 1 variant(s) + 1 fp32 baseline) = 21 rows


## (Optional) Prepare real ImageNet (ILSVRC2012)

The real validation set, which is what timm's published numbers and the TransAxx paper report on. **Not
downloadable here** (image-net.org gate): stage the two files into `docker/datasets/imagenet_data/`
first (`ILSVRC2012_img_val.tar`, 6.3 GB, and `ILSVRC2012_devkit_t12.tar.gz`, 2.5 MB), then set
`PREP_IMAGENET_REAL = True` and run once. It moves the tars under `imagenet/`, uses
`torchvision.datasets.ImageNet` to parse the devkit (`meta.mat` + val ground truth) and lay the
50 000 images out in `imagenet/val/<WNID>/` in the exact class order timm's heads expect, then carves
`imagenet/train_tiny/` for calibration. The first run untars 6.3 GB and takes a few minutes. Use it by setting
`dataset="imagenet"` in `0_configure_sweeps`.

In [6]:
PREP_IMAGENET_REAL = True   # set True ONCE to stage real ILSVRC2012 val (needs the two tars present)

if PREP_IMAGENET_REAL:
    import shutil
    from torchvision.datasets import ImageNet
    root = Path(IMAGENET_DIR)
    real = root / "imagenet"
    real.mkdir(parents=True, exist_ok=True)
    # torchvision.datasets.ImageNet wants the two tars in its root. Move them in if they
    # are still at the top level (instant rename, same filesystem).
    for fn in ("ILSVRC2012_img_val.tar", "ILSVRC2012_devkit_t12.tar.gz"):
        src, dst = root / fn, real / fn
        if src.exists() and not dst.exists():
            shutil.move(str(src), str(dst))
    assert (real / "ILSVRC2012_img_val.tar").exists() or (real / "val").exists(), \
        ("ILSVRC2012_img_val.tar not found — download it + ILSVRC2012_devkit_t12.tar.gz into "
         "docker/datasets/imagenet_data/ first (see the markdown above).")
    # parses meta.mat + val ground truth, untars, lays out val/<WNID>/ in timm's class order.
    # Idempotent: once val/ + meta.bin exist, re-runs are instant.
    print("preparing real ImageNet val (untar 6.3 GB + WNID layout via devkit) — a few minutes the first time ...")
    ImageNet(str(real), split="val")
    val_dir   = real / "val"
    calib_dir = real / "train_tiny"
    CALIB_PER_CLASS = 5
    if not calib_dir.exists():
        for cls in sorted(p for p in val_dir.iterdir() if p.is_dir()):
            dst = calib_dir / cls.name
            dst.mkdir(parents=True, exist_ok=True)
            for img in sorted(cls.glob("*"))[:CALIB_PER_CLASS]:
                shutil.copy(img, dst / img.name)
    n_val   = sum(1 for _ in val_dir.glob("*/*"))
    n_calib = sum(1 for _ in calib_dir.glob("*/*"))
    print(f"real ImageNet ready: val={n_val} imgs / {len(list(val_dir.iterdir()))} classes, "
          f"train_tiny={n_calib} imgs")
    print("(you may delete imagenet/ILSVRC2012_img_val.tar afterwards to reclaim 6.3 GB)")
else:
    print("PREP_IMAGENET_REAL=False — using whatever is already at IMAGENET_DIR/imagenet/.")

preparing real ImageNet val (untar 6.3 GB + WNID layout via devkit) — a few minutes the first time ...
real ImageNet ready: val=50000 imgs / 1000 classes, train_tiny=5000 imgs
(you may delete imagenet/ILSVRC2012_img_val.tar afterwards to reclaim 6.3 GB)


## Data loaders — each model's own transform

A generic ImageNet transform (mean 0.485…/std 0.229…) **under-feeds** ViTs like `vit_small`, which expect
(0.5, 0.5, 0.5) at crop_pct 0.9 / bicubic, and that alone cost ~10 pp on the exact baseline last run. So we
resolve **the model's own eval transform from timm** (`resolve_model_data_config` → `create_transform`) and
build `ImageFolder` loaders with it, per model. The calibration loader is **shuffled (seeded)** so the first
`CALIB_BATCHES` span many classes, much better range estimation than the first-N-classes default.

In [7]:
val_dir   = Path(IMAGENET_DIR) / DATASET_NAME / "val"
calib_dir = Path(IMAGENET_DIR) / DATASET_NAME / "train_tiny"
assert val_dir.exists(),   f"{DATASET_NAME} val not found at {val_dir} — run the Prepare real ImageNet cell"
assert calib_dir.exists(), f"{DATASET_NAME} calib not found at {calib_dir} — run the matching Prepare cell"

def make_transform(model_name):
    # the model's OWN eval transform (correct mean/std/crop/interpolation), via timm
    from timm.data import create_transform
    probe = timm.create_model(model_name, pretrained=False)
    try:
        from timm.data import resolve_model_data_config           # timm >= 0.8
        cfg = resolve_model_data_config(probe)
    except ImportError:
        from timm.data import resolve_data_config                 # older timm
        cfg = resolve_data_config({}, model=probe)
    try:
        tf = create_transform(**cfg, is_training=False)
    except TypeError:
        import torchvision.transforms as T                        # minimal manual fallback
        size = cfg["input_size"][-1]
        crop = cfg.get("crop_pct", 0.9) or 0.9
        interp = {"bicubic": T.InterpolationMode.BICUBIC,
                  "bilinear": T.InterpolationMode.BILINEAR}.get(cfg.get("interpolation", "bicubic"),
                                                                T.InterpolationMode.BICUBIC)
        tf = T.Compose([T.Resize(int(round(size / crop)), interpolation=interp),
                        T.CenterCrop(size), T.ToTensor(),
                        T.Normalize(cfg["mean"], cfg["std"])])
    return tf, cfg

def make_loaders(model_name):
    tf, _ = make_transform(model_name)
    val_ds   = ImageFolder(str(val_dir),   transform=tf)
    calib_ds = ImageFolder(str(calib_dir), transform=tf)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    g = torch.Generator().manual_seed(0)   # seeded shuffle -> calib spans classes, still reproducible
    calib_loader = DataLoader(calib_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return val_loader, calib_loader

# sanity: show the transform we'll actually use for the first model
_tf, _cfg = make_transform(MODELS[0])
print(f"{MODELS[0]}: mean={_cfg.get('mean')} std={_cfg.get('std')} crop_pct={_cfg.get('crop_pct')} "
      f"interp={_cfg.get('interpolation')} input={_cfg.get('input_size')}")
print(f"val={len(list(val_dir.glob('*/*')))} imgs, calib={len(list(calib_dir.glob('*/*')))} imgs "
      f"(calibrating on {CALIB_BATCHES} shuffled batches)")


vit_small_patch16_224: mean=(0.5, 0.5, 0.5) std=(0.5, 0.5, 0.5) crop_pct=0.9 interp=bicubic input=(3, 224, 224)
val=50000 imgs, calib=5000 imgs (calibrating on 8 shuffled batches)


## Model builder + linear-layer wrapping

`build_model` pulls an ImageNet-pretrained ViT from timm. `linear_layers_of` lists the `nn.Linear`
modules we'll approximate (excluding the classifier `head`). `install_multiplier` sets every one to a
single multiplier and JIT-compiles its CUDA kernel, which is the slow first iteration per multiplier name.

In [8]:
def build_model(name):
    return timm.create_model(name, pretrained=True).to(device)

def linear_layers_of(model):
    return [(n, m) for n, m in model.named_modules()
            if (isinstance(m, nn.Linear) or isinstance(m, AdaPT_Linear))
            and "head" not in n and "reduction" not in n]

def install_multiplier(model, mult, n_layers):
    axx_list = [{"axx_mult": mult, "axx_power": 1.0,
                 "quant_bits": 8, "fake_quant": False}] * n_layers
    replace_linear_layers(model, AdaPT_Linear, axx_list, 0, 0,
                          layer_count=[0], returned_power=[0], initialize=True)


## Calibration / evaluation / finetune helpers

`calibrate` runs percentile (99.99) calibration over `CALIB_BATCHES` shuffled batches of `train_tiny`.
`evaluate` does one pass over val, returning top-1 / top-5 / n / runtime + a 1000×1000 confusion matrix.
`finetune` is present for completeness but unused at `RETRAIN_EPOCHS = 0`.


In [9]:
def calibrate(model, loader):
    with torch.no_grad():
        collect_stats(model, loader, num_batches=CALIB_BATCHES, device=device)
        compute_amax(model, method="percentile", percentile=99.99, device=device)

def evaluate(model, loader, n_classes):
    model.eval()
    correct = top5 = total = 0
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    t0 = time.time()
    with torch.no_grad():
        for img, lbl in tqdm(loader, total=len(loader), leave=False):
            img, lbl = img.to(device), lbl.to(device)
            out = model(img)
            _, p5 = out.topk(5, dim=1)
            p1 = p5[:, 0]
            total += lbl.size(0)
            correct += (p1 == lbl).sum().item()
            top5 += (p5 == lbl.unsqueeze(1)).any(dim=1).sum().item()
            for t, p in zip(lbl.cpu().numpy(), p1.cpu().numpy()):
                cm[t, p] += 1
    dt = time.time() - t0
    return 100 * correct / total, 100 * top5 / total, total, dt, cm

def finetune(model, loader, epochs):
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)   # imagenet_eval's ViT recipe
    for e in range(epochs):
        train_one_epoch(model, criterion, optimizer, loader, device, e, 50)


In [10]:
def existing_keys():
    if not CSV_PATH.exists():
        return set()
    df = pd.read_csv(CSV_PATH)
    return {tuple(r) for r in df[["dataset", "model", "multiplier", "retrained_epochs"]]
                                  .itertuples(index=False, name=None)}

def append_row(row):
    new_file = not CSV_PATH.exists()
    with CSV_PATH.open("a", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(COLS)
        w.writerow([row.get(c, "") for c in COLS])

def record(dataset, model_name, mult, retrained_epochs, top1, top5, n, dt, cm):
    rid = uuid.uuid4().hex[:12]
    np.save(cm_path(dataset, model_name, mult, retrained_epochs), cm)
    append_row({
        "run_id": rid,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "dataset": dataset, "model": model_name, "multiplier": mult,
        "retrained_epochs": retrained_epochs,
        "top1": f"{top1:.4f}", "top5": f"{top5:.4f}",
        "n_samples": n, "runtime_s": f"{dt:.1f}", "notes": "",
    })
    return rid


## Sweep

For each model: resolve its transform and build the loaders once, then **evaluate the plain FP32 model**
(one `multiplier=fp32` row, no quantization, no LUT), then for each multiplier build a fresh ViT, wrap
its linear layers, calibrate, eval. The first multiplier per model JIT-compiles the kernel (~30s+). Later
ones reuse it. Rows already in `runs.csv` are skipped, so interrupted runs resume.

In [11]:
done = existing_keys()
print(f"{len(done)} rows already in CSV; will skip those.\n")

for model_name in MODELS:
    val_loader, calib_loader = make_loaders(model_name)   # per-model transform, built once per model

    # FP32 baseline (no quantization, no approximation). Anchors the decomposition:
    # FP32-here vs published real-val FP32 = the dataset's cost; mul8s_acc vs this
    # row = the int8 PTQ cost; the ish rows vs mul8s_acc = the multiplier's cost.
    if (DATASET_NAME, model_name, "fp32", 0) in done:
        print(f"[skip] {DATASET_NAME}/{model_name}/fp32 — present")
    else:
        print(f"\n=== {DATASET_NAME} / {model_name} / fp32 (unquantized baseline) ===")
        m = build_model(model_name)
        top1, top5, n, dt, cm = evaluate(m, val_loader, N_CLASSES)
        rid = record(DATASET_NAME, model_name, "fp32", 0, top1, top5, n, dt, cm)
        print(f"  fp32 baseline  top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")
        del m
        torch.cuda.empty_cache()
        gc.collect()

    for mult in MULTS:
        need_eval    = (DATASET_NAME, model_name, mult, 0) not in done
        need_retrain = RETRAIN_EPOCHS > 0 and \
                       (DATASET_NAME, model_name, mult, RETRAIN_EPOCHS) not in done
        if not (need_eval or need_retrain):
            print(f"[skip] {DATASET_NAME}/{model_name}/{mult} — present")
            continue

        print(f"\n=== {DATASET_NAME} / {model_name} / {mult} ===")
        m = build_model(model_name)
        n_layers = len(linear_layers_of(m))
        install_multiplier(m, mult, n_layers)
        calibrate(m, calib_loader)

        if need_eval:
            top1, top5, n, dt, cm = evaluate(m, val_loader, N_CLASSES)
            rid = record(DATASET_NAME, model_name, mult, 0, top1, top5, n, dt, cm)
            print(f"  eval-only      top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")

        if need_retrain:
            finetune(m, calib_loader, RETRAIN_EPOCHS)
            top1, top5, n, dt, cm = evaluate(m, val_loader, N_CLASSES)
            rid = record(DATASET_NAME, model_name, mult, RETRAIN_EPOCHS, top1, top5, n, dt, cm)
            print(f"  retrained({RETRAIN_EPOCHS}ep) top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")

        del m
        torch.cuda.empty_cache()
        gc.collect()
    del val_loader, calib_loader

from IPython.display import FileLink, display
print("\nDone.")
print(f"Results: {CSV_PATH.resolve()}")
display(FileLink(str(CSV_PATH)))
_df = pd.read_csv(CSV_PATH)
_df[_df.dataset == DATASET_NAME].tail(15)

90 rows already in CSV; will skip those.


=== imagenet / vit_small_patch16_224 / fp32 (unquantized baseline) ===


W0617 09:25:02.298345 128478895546368 _http.py:904] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

  fp32 baseline  top1=81.3800  top5=96.1360  (42.3s)  run_id=19ecade1a9f9

=== imagenet / vit_small_patch16_224 / mul8s_fpga_ish_1 ===


100%|██████████| 8/8 [04:40<00:00, 35.08s/it]
W0617 09:31:16.127418 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.127779 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.127984 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128132 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128276 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128413 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128548 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128678 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128816 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.128946 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 09:31:16.129079 128478895546368 tensor

...

100%|██████████| 8/8 [09:26<00:00, 70.84s/it]
W0617 14:27:03.006711 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.007066 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.007297 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.007467 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.007631 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.007956 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.008120 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.008290 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.008456 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.008606 128478895546368 tensor_quantizer.py:173] Disable HistogramCalibrator
W0617 14:27:03.008762 128478895546368 tensor

blocks.0.attn.qkv.quantizer             : TensorQuantizer(8bit per-tensor amax=8.3834 calibrator=HistogramCalibrator quant)
blocks.0.attn.qkv.quantizer_w           : TensorQuantizer(8bit per-tensor amax=0.2511 calibrator=HistogramCalibrator quant)
blocks.0.attn.proj.quantizer            : TensorQuantizer(8bit per-tensor amax=2.8033 calibrator=HistogramCalibrator quant)
blocks.0.attn.proj.quantizer_w          : TensorQuantizer(8bit per-tensor amax=0.3739 calibrator=HistogramCalibrator quant)
blocks.0.mlp.fc1.quantizer              : TensorQuantizer(8bit per-tensor amax=13.4144 calibrator=HistogramCalibrator quant)
blocks.0.mlp.fc1.quantizer_w            : TensorQuantizer(8bit per-tensor amax=0.1407 calibrator=HistogramCalibrator quant)
blocks.0.mlp.fc2.quantizer              : TensorQuantizer(8bit per-tensor amax=4.6959 calibrator=HistogramCalibrator quant)
blocks.0.mlp.fc2.quantizer_w            : TensorQuantizer(8bit per-tensor amax=0.1279 calibrator=HistogramCalibrator quant)
blocks.

  eval-only      top1=77.9900  top5=93.6520  (1285.9s)  run_id=eaab916b6e35

Done.
Results: /work/results/transaxx/runs.csv


/work/results/transaxx/runs.csv

,run_id,timestamp,dataset,model,multiplier,retrained_epochs,top1,top5,n_samples,runtime_s,notes
96,f4889bb7b4e5,2026-06-17T10:33:24,imagenet,vit_small_patch16_224,mul8s_acc,0,79.092,95.132,50000,355.6,NaN
97,7f1023f6ea80,2026-06-17T10:34:12,imagenet,deit_small_patch16_224,fp32,0,79.856,95.056,50000,40.4,NaN
98,48c6f50b4edd,2026-06-17T10:44:51,imagenet,deit_small_patch16_224,mul8s_fpga_ish_1,0,73.100,91.410,50000,348.3,NaN
99,4313643252fd,2026-06-17T10:55:27,imagenet,deit_small_patch16_224,mul8s_fpga_ish_2,0,73.292,91.660,50000,346.3,NaN
100,b2fe7288f318,2026-06-17T11:06:04,imagenet,deit_small_patch16_224,mul8s_fpga_ish_3,0,73.264,91.576,50000,346.4,NaN
101,1dccb594424d,2026-06-17T11:16:49,imagenet,deit_small_patch16_224,mul8s_fpga_ish_4,0,73.054,91.362,50000,355.0,NaN
102,31aa4d9b01f4,2026-06-17T11:27:29,imagenet,deit_small_patch16_224,mul8s_fpga_ish_5,0,73.078,91.454,50000,349.0,NaN
103,a2984bbf45c2,2026-06-17T11:38:23,imagenet,deit_small_patch16_224,mul8s_acc,0,73.248,91.636,50000,364.4,NaN
104,b128fb715614,2026-06-17T11:40:48,imagenet,vit_base_patch16_224,fp32,0,85.104,97.526,50000,124.9,NaN
105,9722bf3c93fb,2026-06-17T12:13:02,imagenet,vit_base_patch16_224,mul8s_fpga_ish_1,0,78.064,93.650,50000,1366.9,NaN
